##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma Text Fine-tuning Tutorial - Translator of Old Korean Literature

## Set the Goal of Tuning

This tutorial demonstrates how to fine-tune Gemma to translate Classical Korean into Modern Korean. Fine-tuning is essential when a base model lacks specific linguistic domain knowledge.

The Korean alphabet, or Hangul, has undergone changes over time, resulting in several letters no longer used in modern Korean. These obsolete letters include:

1. ㆍ (Arae-a): This dot vowel represents a short 'a' sound.
2. ㆆ (Yeorin-hieut): Pronounced as a 'light h,' akin to a softer version of the English 'h.'
3. ㅿ (Bansiot): Represents the 'z' sound.
4. ㆁ (Yet-ieung): A velar nasal sound comparable to 'ng' in the word 'sing.'

Reading older literature is a challenge even for native Korean speakers due to the these obsolete letters and the lack of word spacing.

However, by fine-tuning Gemma, it becomes possible to create a translator that can aid in understanding and bridging the gap between contemporary and archaic Korean.

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/tutorials/Translator_of_Old_Korean_Literature.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

Note: This guide was created to run on a Google colaboratory account using a NVIDIA T4 GPU with 16GB and [Gemma 4 E2B (IT)](https://huggingface.co/google/gemma-4-E2B-it), but can be adapted to run on bigger GPUs and bigger models.

## Setup development environment

The first step is to install Hugging Face Libraries, including TRL, and datasets to fine-tune open model, including different RLHF and alignment techniques.

In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Transformers
%pip install "transformers>=5.10.1"

# Install Hugging Face libraries
%pip install datasets accelerate evaluate bitsandbytes trl "peft>=0.19.0" protobuf sentencepiece

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#%pip install flash-attn

_Note: If you are using a GPU with Ampere architecture (such as NVIDIA L4) or newer, you can use Flash attention. Flash Attention is a method that significantly speeds computations up and reduces memory usage from quadratic to linear in sequence length, leading to acelerating training up to 3x. Learn more at [FlashAttention](https://github.com/Dao-AILab/flash-attention/tree/main)._

You need a valid Hugging Face Token to publish your model. If you are running inside a Google Colab, you can securely use your Hugging Face Token using the Colab secrets otherwise you can set the token as directly in the `login` method. Make sure your token has write access too, as you push your model to the Hub during training.

In [2]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()

## Build Your Own Dataset

This tutorial uses a dataset derived from "HongGildongJeon (홍길동전)", a famous Joseon Dynasty novel.

Here's [the dataset](https://huggingface.co/datasets/bebechien/HongGildongJeon). The [original source](https://ko.wikisource.org/wiki/%ED%99%8D%EA%B8%B8%EB%8F%99%EC%A0%84_36%EC%9E%A5_%EC%99%84%ED%8C%90%EB%B3%B8) is in public domain. You will use a [modern translation](https://ko.wikisource.org/wiki/%ED%99%8D%EA%B8%B8%EB%8F%99%EC%A0%84_36%EC%9E%A5_%EC%99%84%ED%8C%90%EB%B3%B8/%ED%98%84%EB%8C%80%EC%96%B4_%ED%95%B4%EC%84%9D) in a [creative commons license](https://creativecommons.org/licenses/by-sa/4.0/), translated by `직지프로`.

To prepare the data for Gemma, we format the text into conversational message, defining roles for the `system`, `user` (Old Korean), and `assistant` (Modern Korean). The following code handles loading, formatting, and splitting the dataset into training and testing sets.

```
[
  {"role": "system", "content": "Translate Classical Korean into Modern Korean."},
  {"role": "user", "content": "됴션국셰둉ᄃᆡ왕즉위십오연의홍희문밧긔ᄒᆞᆫᄌᆡ상이잇스되"},
  {"role": "assistant", "content": "조선국 세종대왕 즉위 십오년에 홍회문 밖에 한 재상이 있으되,"},
]
```

> NOTE: korean text means, In the fifteenth year of the reign of King Sejong of Joseon, there was a prime minister outside Honghoemun Gate.

In [1]:
from datasets import load_dataset

ds = load_dataset(
    "bebechien/HongGildongJeon",
    split="train",
)
print(ds)

data = ds.with_format(
    "np", columns=["original", "modern translation"], output_all_columns=False
)
train = []

system_message = "Translate Classical Korean into Modern Korean."

def create_conversation(sample, idx):
  return {
    "messages": [
      {"role": "system", "content": system_message},
      {"role": "user", "content": sample["original"]},
      {"role": "assistant", "content": sample["modern translation"]}
    ]
  }

dataset = load_dataset("bebechien/HongGildongJeon", split="train")
dataset = dataset.map(create_conversation, with_indices=True, remove_columns=dataset.features)
dataset = dataset.train_test_split(test_size=0.2, shuffle=False)

print(dataset["train"][0])
print(dataset["test"][0])

Dataset({
    features: ['original', 'modern translation'],
    num_rows: 447
})
{'messages': [{'role': 'system', 'content': 'Translate Classical Korean into Modern Korean.'}, {'role': 'user', 'content': '됴션국셰둉ᄃᆡ왕즉위십오연의홍희문밧긔ᄒᆞᆫᄌᆡ상이잇스되셩은홍이요명은문이니위인이쳥염강직ᄒᆞ여덩망이거록ᄒᆞ니당셰의영웅이라일직용문의올나벼살이할림의쳐ᄒᆞ엿더니명망이됴졍의읏듬되ᄆᆡ젼하그덕망을승이녀긔ᄉᆞ벼살을도도와이조판셔로좌으졍을ᄒᆞ이시니승상이국은을감동ᄒᆞ야갈츙보국ᄒᆞ니ᄉᆞ방의일이업고도젹이업스ᄆᆡ시화연풍ᄒᆞ여나라이ᄐᆡ평ᄒᆞ더라'}, {'role': 'assistant', 'content': '조선국 세종대왕 즉위 십오년에 홍회문 밖에 한 재상이 있으되, 성은 홍이요, 명은 문이니, 위인이 청렴강직하여 덕망이 거룩하니 당세의 영웅이라. 일찍 용문에 올라 벼슬이 한림에 처하였더니 명망이 조정의 으뜸 되매, 전하 그 덕망을 승히 여기사 벼슬을 돋우어 이조판서로 좌의정을 하게 하시니, 승상이 국은을 감동하여 갈충보국하니 사방에 일이 업고 도적이 없으매 시화연풍하여 나라가 태평하더라.'}]}
{'messages': [{'role': 'system', 'content': 'Translate Classical Korean into Modern Korean.'}, {'role': 'user', 'content': '길동이탄식왈나ᄂᆞᆫ쳔지간블효ᄌᆡ라ᄂᆡ본ᄃᆡ이곳ᄉᆞᄅᆞᆷ이아니라조션국홍승상의쳔쳡소ᄉᆡᆼ이라집안의쳔ᄃᆡᄌᆞ심ᄒᆞ고조졍으도ᄎᆞᆷ예치못ᄒᆞᄆᆡ장부을희을참지못ᄒᆞ여부모을ᄒᆞ직ᄒᆞ고이곳의와은신ᄒᆞ여시나부모의긔후을ᄉᆞ모ᄒᆞ더니오날날쳔문을살피니부친의유명ᄒᆞ신명이불구의셰상을이별ᄒᆞ실지라ᄂᆡ몸이만리외예잇셔밋쳬득달치못ᄒᆞ게되니ᄉᆡᆼ젼의부친안젼의ᄇᆡ옵지못ᄒᆞ게되오ᄆᆡ글노스러ᄒᆞ노라'}, {'role': 'assistant', 'conte

## Fine-tune Gemma using TRL and the SFTTrainer

You are now ready to fine-tune your model. Hugging Face TRL [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) makes it straightforward to supervise fine-tune open LLMs. The `SFTTrainer` is a subclass of the `Trainer` from the `transformers` library and supports all the same features, including logging, evaluation, and checkpointing, but adds additional quality of life features, including:

* Dataset formatting, including conversational and instruction formats
* Training on completions only, ignoring prompts
* Packing datasets for more efficient training
* Parameter-efficient fine-tuning (PEFT) support including QLoRA

The following code loads the Gemma model and tokenizer from Hugging Face and initializes the tuning configuration.

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "google/gemma-4-E2B-it" # @param ["google/gemma-4-E2B-it","google/gemma-4-E4B-it"] {"allow-input":true}

# Check if GPU supports bfloat16
if torch.cuda.is_bf16_supported():
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

# Define model init arguments
model_kwargs = dict(
    dtype=torch_dtype, # What torch dtype to use
    device_map="auto", # Let torch decide how to load the model
)

# BitsAndBytesConfig: Enables 4-bit quantization to reduce model size/memory usage
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_storage=torch_dtype,
)

# Load model and processor
model = AutoModelForMultimodalLM.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)

# NOTE: You should call the prepare_model_for_kbit_training() function to preprocess the quantized model for training.
# On T4, we are skipping this step purely due to VRAM limitation and for a quick demonstration.
if (torch.cuda.get_device_properties(0).total_memory/1024**3) > 16:
    model = prepare_model_for_kbit_training(model)

# Training Configurations
output_dir = "gemma-old-korean" # @param {"type":"string"}
token_limit = 1024 # @param {"type":"integer"}
train_epoch = 5 # @param {"type":"integer"}
batch_size = 1 # @param {"type":"integer"}
learning_rate = 2e-4 # @param {"type":"number"}
lora_rank = 16 # @param {"type":"integer"}

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Set the Evaluation Plan

To measure progress quantitatively, we use a character-by-character similarity ratio. This is implemented using `difflib.SequenceMatcher`, which calculates a score between 0.0 and 1.0 based on how closely the generated modern Korean matches the human translation. This metric allows us to objectively compare the model's performance before and after tuning.

In [3]:
import difflib
from transformers import pipeline, GenerationConfig
config = GenerationConfig.from_pretrained(model_id)
config.max_new_tokens = token_limit
config.disable_compile = True

def check_success_rate(full_test: bool = False):
    total_similarity = 0.0
    sample_count = 0

    pipe = pipeline("text-generation", model=model, tokenizer=processor.tokenizer)

    for idx, item in enumerate(dataset["test"]):
        if not full_test and idx >= 3:
            break

        messages = item["messages"][:2]
        outputs = pipe(messages, return_full_text=False, generation_config=config)
        text = outputs[0]["generated_text"].removesuffix("<turn|>")
        original_text = item["messages"][2]["content"]

        # Calculate character-by-character similarity ratio (0.0 to 1.0)
        similarity = difflib.SequenceMatcher(
            None, text, original_text
        ).ratio()

        print(f"Original: {original_text}")
        print(f"Generated: {text}")
        print(f"Similarity Score: {similarity:.2%}")

        total_similarity += similarity
        sample_count += 1

    if sample_count > 0:
        avg_score = total_similarity / sample_count
        print(f"\nAverage Similarity Score: {avg_score:.2%}")
    else:
        print("\nNo samples processed.")


## Verify Model Capability (Before Fine-tuning)

Before starting the fine-tuning process, it is essential to demonstrate that the base model lacks the specific capability for this archaic translation task. We use a test script to generate translations from the base model and compare them against the ground truth.

As seen in the results below, the base model typically provides an interpretive explanation or a very literal (and often incorrect) reconstruction, resulting in very low initial similarity scores.


In [4]:
check_success_rate()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Original: 길동이 탄식 왈,
"나는 천지간 불효자라. 나는 본디 이 곳 사람이 아니라, 조선국 홍승상의 천첩소생이라. 집안의 천대 자심하고, 조정에도 참여치 못하매, 장부 울회를 참지 못하여 부모를 하직하고 이곳에 와 은신하였으나 부모의 기후를 사모하더니, 오늘날 천문을 살피니 부친의 유명하신 명이 불구에 세상을 이별하실지라. 내 몸이 만리 외에 있어 미처 득달치 못하게 되니 생전의 부친 안전에 뵙지 못하게 되오매 그것을 슬퍼하노라."
Generated: Here is the translation of the Classical Korean text into Modern Korean:

**Original Classical Korean:**
길동이탄식왈나ᄂᆞᆫ쳔지간블효ᄌᆡ라ᄂᆡ본ᄃᆡ이곳ᄉᆞᄅᆞᆷ이아니라조션국홍승상의쳔쳡소ᄉᆡᆼ이라집안의쳔ᄃᆡᄌᆞ심ᄒᆞ고조졍으도ᄎᆞᆷ예치못ᄒᆞᄆᆡ장부을희을참지못ᄒᆞ여부모을ᄒᆞ직ᄒᆞ고이곳의와은신ᄒᆞ여시나부모의긔후을ᄉᆞ모ᄒᆞ더니오날날쳔문을살피니부친의유명ᄒᆞ신명이불구의셰상을이별ᄒᆞ실지라ᄂᆡ몸이만리외예잇셔밋쳬득달치못ᄒᆞ게되니ᄉᆡᆼ젼의부친안젼의ᄇᆡ옵지못ᄒᆞ게되오ᄆᆡ글노스러ᄒᆞ노라

**Modern Korean Translation:**

"길동이 탄식(탄식하는 소리)을 하며 말하기를, '나는 지금 집안에 효도하는 자라고 생각한다. 이곳 사람들은 아니며, 조선 국왕의 승상(재상) 소생이라 집안의 짐을 지심하고 조정에서도 짐을 감당하지 못하여 장부를 펴지 못해 부모를 잃게 되었다. 이곳의 와은신(와은신이시니)하여시나 부모의 뒤를 섬기었더니, 날마다 문을 살피니 부친의 유명한 신명이 불구의 세상을 이별하실지라 몸이 만리 밖에 있어 득달같이 달리지 못하게 되니, 선대의 부친을 편안하게 모시지 못하게 되니 글이 슬프다.'"

---

**A more natural, slightly polished interpretation:**

"길동이 탄식하며 말했다. '나는 지금 집안에 효도하는 사람이라고 생각한다. 이곳 사람들은 아니요, 조선 국왕의 승상

## Training

Fine-tuning requires setting up the development environment and configuring Parameter-Efficient Fine-Tuning (PEFT) using LoRA. It is important to experiment with multiple values for LoRA rank, learning rate, and epochs to find the optimal balance for your specific dataset.

In [4]:
from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=lora_rank,
    lora_dropout=0.05,
    r=lora_rank,
    bias="none",
    # no target_modules — PEFT's Gemma 4 defaults scope to the LM layers
    task_type="CAUSAL_LM",
)

Before you can start your training, you need to define the hyperparameter you want to use in a `SFTConfig` instance.

In [5]:
import torch
from trl import SFTConfig

total_steps = (len(dataset['train']) // batch_size) * train_epoch
calculated_warmup_steps = int(total_steps * 0.03)

args = SFTConfig(
    output_dir=output_dir,                  # directory to save and repository id
    max_length=token_limit*2,               # max length for model and packing of the dataset
    num_train_epochs=train_epoch,           # number of training epochs
    per_device_train_batch_size=batch_size, # batch size per device during training
    per_device_eval_batch_size=batch_size,  # batch size per device during evaluation
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every 1 steps
    save_strategy="epoch",                  # save checkpoint every epoch
    eval_strategy="epoch",                  # evaluate checkpoint every epoch
    learning_rate=learning_rate,            # learning rate
    lr_scheduler_type="cosine",             # learning rate scheduler
    warmup_steps=max(calculated_warmup_steps, 10),
    push_to_hub=True,                       # push model to hub
    dataset_kwargs={"skip_prepare_dataset": True}, # important for collator
    remove_unused_columns = False,                 # important for collator
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",      # track loss, not accuracy
    greater_is_better=False,                # Lower loss is better
)

# Data collator
def collate_fn(examples):
    texts = []

    for example in examples:
        full_text = processor.apply_chat_template(
            example["messages"], add_generation_prompt=False, tokenize=False
        )
        texts.append(full_text.strip())

    # Tokenize the texts and process the audios
    batch = processor(text=texts, return_tensors="pt", padding=True)

    # The labels are the input_ids, and we mask the padding tokens and audio tokens in the loss computation
    labels = batch["input_ids"].clone()

    target_tokens = [
        processor.tokenizer.convert_tokens_to_ids("<|turn>"),
        processor.tokenizer.convert_tokens_to_ids("model"),
        processor.tokenizer.convert_tokens_to_ids("\n")
    ]
    target_len = len(target_tokens)

    for i in range(labels.size(0)):
        row_tokens = batch["input_ids"][i].tolist()

        # Find where the assistant block begins
        assistant_start_idx = None
        for idx in range(len(row_tokens) - target_len + 1):
            if row_tokens[idx : idx + target_len] == target_tokens:
                # We want to keep loss calculation on the assistant transcription tokens,
                # so we move the index right past the assistant header ('<|turn>\nmodel\n')
                assistant_start_idx = idx + target_len
                break

        if assistant_start_idx is not None:
            # Mask everything from index 0 up to the start of the actual Japanese text response
            labels[i, :assistant_start_idx] = -100
        else:
            # Fallback safety: if template matching fails for an anomalous row, mask padding anyway
            print("WARNING: maybe the sample is too long, try to increase `token_limit` value.")
            labels[i, labels[i] == processor.tokenizer.pad_token_id] = -100

        """
        # --- DEBUG PRINT CODE ---
        print(f"\n--- Example {i} (Split index: {assistant_start_idx}) ---")
        debug_string = []
        for token_id, label_id in zip(row_tokens, labels[i].tolist()):
            # Decode token by token so we can see exactly what is masked
            decoded_token = processor.tokenizer.decode([token_id])

            if label_id == -100:
                # Red text for masked tokens (ANSI Escape Code)
                debug_string.append(f"\033[91m{decoded_token}\033[0m")
            else:
                # Green text for active loss tokens
                debug_string.append(f"\033[92m{decoded_token}\033[0m")

        print("".join(debug_string))
        # ------------------------
        """


    # Mask tokens for not being used in the loss computation
    labels[labels == processor.tokenizer.pad_token_id] = -100

    batch["labels"] = labels
    return batch

/tmp/ipykernel_58102/1271903062.py:7: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(


A critical component of this setup is the `collate_fn`. This function ensures that the model only learns from the 'assistant' completions by masking the user prompts and system messages in the loss calculation. This forces the model to focus strictly on generating the translation.

You now have every building block you need to create your `SFTTrainer` to start the training of your model.

In [6]:
from transformers import EarlyStoppingCallback
from trl import SFTTrainer

# Create Trainer object
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    data_collator=collate_fn,
)

Start training by calling the `train()` method.

In [7]:
# Start training, the model will be automatically saved to the Hub and the output directory
trainer.train()

print("Best checkpoint loaded:", trainer.state.best_model_checkpoint)
print("Best metric score:", trainer.state.best_metric)

# Save the final model again to the Hugging Face Hub
trainer.save_model()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.788810,0.869182,0.801379,0.799568,109204.000000
2,0.541909,0.762507,0.704137,0.825826,218408.000000
3,0.592273,0.778713,0.479882,0.830531,327612.000000
4,0.584903,0.818485,0.408707,0.831316,436816.000000


Best checkpoint loaded: gemma-old-korean/checkpoint-714
Best metric score: 0.7625073790550232


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-korean/training_args.bin: 100%|##########| 5.65kB / 5.65kB            

  ...adapter_model.safetensors:  59%|#####8    | 6.28MB / 10.7MB            

  ...old-korean/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

## Summary

### After Fine-tuning Results

The following results demonstrate the significant improvement in translation accuracy and similarity scores after applying the SFT training process.

In [8]:
check_success_rate(True)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Original: 길동이 탄식 왈,
"나는 천지간 불효자라. 나는 본디 이 곳 사람이 아니라, 조선국 홍승상의 천첩소생이라. 집안의 천대 자심하고, 조정에도 참여치 못하매, 장부 울회를 참지 못하여 부모를 하직하고 이곳에 와 은신하였으나 부모의 기후를 사모하더니, 오늘날 천문을 살피니 부친의 유명하신 명이 불구에 세상을 이별하실지라. 내 몸이 만리 외에 있어 미처 득달치 못하게 되니 생전의 부친 안전에 뵙지 못하게 되오매 그것을 슬퍼하노라."
Generated: 길동이 탄식 왈,
"나는 천지간에 불효져나면 둔데 이곳 사람이 아니로다. 조선 국홍승상의 천천소생이라. 집안에 천대자심하고 정원도 참예치 못함에 장부를 뒤에 참지 못하여 부모를 학대하고 이 곳에 와서 은신하여 있으나 부모의 기후를 섬모하더니, 오늘날 천문을 살피니 부친의 유명하신 명 이 불괴세상을 이별할지라. 내가 몸이 만리 외에 있어 미천득달치 못하게 되니 생전 내 부친 안장에 배없이 못하게 되오매 글러러 수러하노라."
Similarity Score: 78.83%
Original: 백씨 듣고 내심에 탄복 왈,
"그 근본을 감추지 아니하니 장부로다!"
하고, 재삼 위로하더라.
Generated: 백씨들고 내심에 탄복 왈,
"그 근본을 감초지 아니하니 장부로다 하고 제삼위에로 하더라."
Similarity Score: 83.81%
Original: 이때에 길동이 군사를 거느리고 일봉산에 들어가 산기를 살펴 명당을 정하고, 날을 가리어 역사를 시작하여 좌우 산곡과 분묘를 능과 같이 하고 돌아와 모든 군사를 불러 왈,
"모월 모일 대선 한척을 준비하여 조선 서강에 와 기다리라."
하고,
"부모를 모셔 올 것이니 미리 알아 거행하라."
한대, 모든 군사 청령하고 물러가 거행하니라.
Generated: 이때 예길동이 군사를 거느리고 일봉산에 들어가 산기울을 살펴보니 명당을 정하고 날을 갈이려 역사를 시작하여 좌우 산곡과 분묘를 능과 같은 하고, 돌아와 모든 군사를 불러 왈, "모월 모일대서 한책을 준비하여 조서

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Original: 길동이 부인과 그 모친을 위로한 후 그 형장을 대하여 왈,
"소제 그간은 산중에 은거하여 지리를 잠심하여 대감의 말년유택을 정한 곳이 있사옵더니, 알지 못하겠구나! 이미 소점이 있사옵나이까?"
Generated: 길동이 부인과 그 모친을 위로한 후에 그 형장을 대하여 왈,
"소제 길관은 산중에 은거하여 지리를 잡심하며 대감의 말년 유택을 정한 곳에 있사오니, 어찌 못하게 라임에 소체가 있사오니 있거라."
Similarity Score: 79.07%
Original: 그 형이 이 말을 듣고 더욱 반겨 아직 정하지 못한 말을 설화하고,
Generated: 그 형이 이 말을 듣고 더욱 반겨야 적정치 못한 말을 설화하고,
Similarity Score: 88.89%
Original: 제인이 모여 밤이 새도록 정회를 풀고, 이틑날 길동이 그 형을 모시고 한 곳에 이르러 가르켜 왈,
"이곳이 소제의 정한 땅이로소이다."
Generated: 제인이 모화 밤이 맛도록 경회를 베풀고 있듯이 날 길이 동이 그 형을 모시고 한 곳에 이러가르쳐 왈,
"이곳에 소저지정한 병이 로소이다."
Similarity Score: 81.82%
Original: 길현이 사면을 살펴보니, 중중한 석각이 험악하고, 누누한 고총이 수없는지라. 심내에 불합하여 왈,
"소제의 높은 소견을 알지 못하되 내 마음은 이곳에 모실 생각이 없으니 다른 땅을 점복하라."
Generated: 길현이 사면을 살펴보니 중중한석각이 심악하고 누누한 고총이 수업나는지라. 심내에 블합하여 왈,
"소제의 놉은 소견은 아지 못하되, 네 마음은 이고 대모 슬생각이 입스리느니라.
Similarity Score: 79.61%
Original: 길동이 거짓 탄식 왈,
"이땅이 비록 이러하오나 누대 장상지지어늘 형장의 소견이 불합하오니 개탄이로다!"
하고,
Generated: 길동이 거짓 탄식 왈,
"이 밖에 비록 이러하오나 누대 장상지직이 연만은 형장의 소견이 불합하오니, 개탄이로이다 하고,
Similarity Sco

Try texts from other Old Korean Literature.

In [10]:
test_list = [
    "나랏말ᄊᆞ미듕귁에달아문ᄍᆞ와로서르ᄉᆞᄆᆞᆺ디아니ᄒᆞᆯ쎄",
    "ᄃᆡ작ᄒᆞ여그ᄭᅩᆺ치흣터지거ᄂᆞᆯ",
    "금두겁이품의드러뵈니일졍ᄌᆡᄌᆞᄅᆞᆯ나흐리로다ᄒᆞ더니과연그ᄃᆞᆯ부터잉ᄐᆡᄒᆞ여십삭이차니",
    "이ᄯᆡᄂᆞᆫᄉᆞ월초팔일이라이날밤의오ᄉᆡᆨ구룸이집을두루고향ᄂᆡ진동ᄒᆞ며션녀ᄒᆞᆫᄡᅣᆼ이촉을들고드러와김ᄉᆡᆼᄃᆞ려니르ᄃᆡ",
    "ᄌᆡ히길너텬졍을어긔지말으소셔이아희ᄇᆡ필은낙양니샹셔집아ᄌᆡ니",
    "천세우희미리정ᄒᆞ샨한수북에누인개국ᄒᆞ샤복년이ᄀᆞᇫ업스시니",
]

pipe = pipeline("text-generation", model=model, tokenizer=processor.tokenizer)

for item in test_list:
    messages = [
        {"role":"system", "content": system_message},
        {"role":"user", "content": item}
    ]
    output = pipe(messages, return_full_text=False, generation_config=config)
    print(output[0]['generated_text'].removesuffix("<turn|>"))


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


나라 말씀이 중국에 달아 문짝자와서라 맡디아니할쎄,
대작하여 그곳에 터져지거늘,
금두겁이 품에 들어 보이니 일제자재를 나흘이 로다 하더니, 과연 그달부터 인정하여 십사(十三) 시까지니,
이때는 사월 초팔일이라. 나일 밤에 오색구름이 집을 두루 고향에 진동하며 선녀한 방에 촉을 들고 돌아와 김생달려 이르되,
제히른 태정을 어기지말소서 이아희패필은 낙양에 상서집어지니,
천세 우희 밀리 정하산
한 수북에 누인 개국하사 복년이 가정에서 시니,


By following this structured workflow:
1. Defining a clear goal
2. Preparing a high-quality conversational dataset
3. Verifying the gap with the base model
4. Applying completions-only training

You successfully tuned Gemma into a capable translator for Old Korean literature. This bridge between archaic and modern language preserves cultural heritage while making it accessible to modern readers.
